<a href="https://colab.research.google.com/github/MarcinPlodzien/deep-learning-for-scientific-discovery/blob/main/notebooks/lecture-07/exercise-enso.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Forecasting El Niño

*Deep Learning for Scientific Discovery, Jagiellonian University. Author: Marcin Płodzień.*

Every few years the surface of the eastern and central tropical Pacific Ocean becomes unusually warm for many months. Fishermen on the coast of Peru noticed it long ago, since the warm water appeared around Christmas and the fish disappeared, and they called it El Niño, the child. The opposite phase, an unusually cold ocean, is called La Niña. Together with the Southern Oscillation, an oscillation of air pressure between the eastern and the western Pacific, the two phases form the El Niño–Southern Oscillation (ENSO), the strongest year-to-year fluctuation of the climate on Earth. It is a coupled oscillation of ocean and atmosphere. The trade winds push warm surface water to the west, a warm eastern Pacific weakens the trade winds, and waves inside the ocean carry the change across the basin over months.

ENSO changes the weather far from the Pacific. Strong El Niño years bring heavy rain to the coast of Peru and Ecuador, droughts to Indonesia and Australia, and change the monsoon rains of India and the winters of North America. A forecast of ENSO several months ahead is therefore one of the most useful climate forecasts, and weather services issue it every month with physical models of ocean and atmosphere and with statistical models of the kind used in the lecture.

We explore the Niño 3.4 index using its time plot, histogram, autocorrelation function, power spectrum and delay plot, with supplied code for each calculation. We then forecast the monthly index up to 20 months ahead with the models of the lecture, from the autoregressive (AR), moving-average (MA) and combined (ARMA) models through the nonlinear vector autoregression (NVAR) to a multilayer perceptron (MLP), an Elman recurrent neural network (RNN) and a long short-term memory network (LSTM). For every model we plot the forecasts against the true values at the horizons τ = 1, 5 and 20 months, the capacity C(τ) against τ, and collect the total capacities in one table. The reader implements the models. The analyses of central England weather and sunspot numbers apply the same calculations to two other records. Their notebooks include implementations of the calculations, and the lecture notebooks explain each model in detail.

> **Google Colab execution**
> 1. Go to <https://colab.research.google.com> → *File → Upload notebook*. The setting `JAX_PLATFORMS=cpu` selects CPU execution, which takes a few minutes.
> 2. `jax`, `flax`, `optax`, `statsmodels`, `scikit-learn`, `scipy`, `pandas` and `matplotlib` are pre-installed in Colab.
> 3. The data are downloaded from the Physical Sciences Laboratory of NOAA during execution, so an internet connection is needed.
> 4. Run the cells *in order*. Later cells use variables defined earlier.

In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"     # run JAX on the CPU (set before importing jax)
import time, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy.stats
import scipy.signal
import jax, jax.numpy as jnp
import optax
import flax
from flax import nnx
import sklearn
from sklearn.neural_network import MLPRegressor
import statsmodels
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf
warnings.filterwarnings("ignore", module="statsmodels")
warnings.filterwarnings("ignore", category=sklearn.exceptions.ConvergenceWarning)

plt.rcParams.update({"font.size": 15, "axes.titlesize": 16, "axes.labelsize": 15, "xtick.labelsize": 13,
                     "ytick.labelsize": 13, "legend.fontsize": 13, "figure.titlesize": 17,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})
UJBlue, UJNavy, DSOrange, UJGray = "#005CA7", "#002B5C", "#D35A05", "#5A6270"
os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)
t_start = time.time()
print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__,
      "| statsmodels", statsmodels.__version__, "| scikit-learn", sklearn.__version__)

<hr/>
## 1. The Niño 3.4 index

The state of ENSO is measured by the temperature of the sea surface in a box of the central Pacific, the Niño 3.4 region between 5° south and 5° north and between 170° west and 120° west. The sea surface temperature is measured by ships, by drifting and moored buoys and, since the 1980s, by infrared radiometers on satellites. The HadISST data set of the Met Office Hadley Centre combines these measurements into monthly maps of the whole ocean on a grid of one degree, back to 1870, with statistical methods that fill the gaps of the early, sparse observations [N. A. Rayner *et al.*, J. Geophys. Res. **108**, 4407 (2003), [doi:10.1029/2002JD002670](https://doi.org/10.1029/2002JD002670)].

The index is an anomaly. For every month, the average sea surface temperature over the Niño 3.4 box is computed, and the mean of the same calendar month over the reference period 1981–2010 is subtracted,
$$s_k=T_k-\overline T_{\text{month}(k)} , \tag{1}$$
where $T_k$ is the box average of month k and $\overline T_{\text{month}(k)}$ the average of all Januaries, Februaries and so on of the reference period. The subtraction removes the seasonal cycle, so the index measures how much warmer or colder the ocean is than usual for that time of the year, in °C. One sample is one such monthly anomaly. A value above +0.5 °C sustained over several months marks El Niño conditions and a value below −0.5 °C La Niña conditions.

The Physical Sciences Laboratory of NOAA computes the index from HadISST and distributes it as a text file with one row per year and twelve monthly values. Months that are not yet available are marked by −99.99 and dropped.

> Source: Niño 3.4 sea surface temperature anomaly from HadISST, NOAA Physical Sciences Laboratory, <https://psl.noaa.gov/data/timeseries/month/>, data file <https://psl.noaa.gov/data/timeseries/month/data/nino34.long.anom.data>. HadISST: Met Office Hadley Centre, Rayner *et al.* (2003).

In [ ]:
ENSO_URL = "https://psl.noaa.gov/data/timeseries/month/data/nino34.long.anom.data"
ENSO_FILE = "data/nino34.long.anom.data"
if not os.path.exists(ENSO_FILE):
    urllib.request.urlretrieve(ENSO_URL, ENSO_FILE)
rows = []
for line in open(ENSO_FILE).read().splitlines()[1:]:
    tok = line.split()
    if len(tok) == 13 and tok[0].isdigit():
        rows += [(int(tok[0]), m + 1, float(v)) for m, v in enumerate(tok[1:])]
enso_all = pd.DataFrame(rows, columns=["year", "month", "anomaly"])
enso = enso_all[enso_all.anomaly > -99].reset_index(drop=True)
t_enso = (enso.year + (enso.month - 0.5) / 12).to_numpy()
nino = enso["anomaly"].to_numpy(float)
DT = 1.0 / 12.0
print(f"{len(enso)} months, {enso.year.iloc[0]}-{enso.month.iloc[0]:02d} to {enso.year.iloc[-1]}-{enso.month.iloc[-1]:02d}, "
      f"{len(enso_all) - len(enso)} months not yet available")
print(enso.tail(4).to_string(index=False))

The tropical Pacific is drawn schematically, without measured data, in normal conditions and during El Niño. In normal years the trade winds blow from east to west along the equator and pile up warm surface water in the west, near Indonesia, while cold water rises from the deep ocean near South America. During El Niño the trade winds weaken, the warm water spreads east, and the box of the Niño 3.4 index, between 5° south and 5° north and between 170° west and 120° west, becomes warmer than usual. The index is the temperature anomaly of this box.

In [ ]:
# schematic of the tropical Pacific: normal conditions and El Nino, with the Nino 3.4 box
from matplotlib.patches import Rectangle
lon = np.linspace(120, 280, 400)                      # degrees east: 120E ... 80W
lat = np.linspace(-20, 20, 120)
LON, LAT = np.meshgrid(lon, lat)
band = np.exp(-(LAT / 9) ** 2)
sst_normal = 26 + 3.5 * band * np.tanh((200 - LON) / 30) - 1.5 * band * np.exp(-((LON - 270) / 15) ** 2)
sst_nino = sst_normal + 2.5 * band * np.exp(-((LON - 230) / 35) ** 2)
fig, axs = plt.subplots(2, 1, figsize=(13, 7.5), layout="constrained", sharex=True)
for ax, field, title, wind in ((axs[0], sst_normal, "normal conditions", 1.0), (axs[1], sst_nino, "El Niño", 0.35)):
    im = ax.pcolormesh(LON, LAT, field, cmap="RdYlBu_r", vmin=22, vmax=31, shading="auto")
    ax.add_patch(Rectangle((190, -5), 50, 10, fill=False, lw=2.5, color="k"))
    ax.text(215, 6.5, "Niño 3.4", ha="center", fontsize=13)
    for x0 in (250, 205, 160):
        ax.annotate("", xy=(x0 - 25 * wind, -12), xytext=(x0, -12),
                    arrowprops=dict(arrowstyle="->", lw=1 + 2 * wind, color=UJNavy))
    ax.text(270, -16.5, "trade winds", ha="right", color=UJNavy, fontsize=12)
    ax.text(125, 15, "Indonesia,\nAustralia", fontsize=12, va="top")
    ax.text(277, 15, "South\nAmerica", fontsize=12, va="top", ha="right")
    ax.set_title(title); ax.set_ylabel("latitude [°]")
axs[1].set_xticks([120, 160, 200, 240, 280], ["120°E", "160°E", "160°W", "120°W", "80°W"])
axs[1].set_xlabel("longitude")
fig.colorbar(im, ax=axs, label="sea surface temperature [°C], schematic")
plt.show()

<hr/>
## 2. Statistical properties of a record

The statistical properties of a record indicate which forecasts can be expected to work. We examine its distribution, temporal correlations and periodic components before fitting a model.

The mean and the standard deviation give the level and the typical size of the fluctuations, and the histogram shows the distribution of the values, whether it is symmetric, bounded or has long tails.

The autocorrelation function measures the linear dependence between values separated in time. For a record $s_1,\dots,s_N$ with mean $\bar s$ it is
$$\rho(m)=\frac{\sum_{k=1}^{N-m}(s_k-\bar s)(s_{k+m}-\bar s)}{\sum_{k=1}^{N}(s_k-\bar s)^2} , \tag{2}$$
the correlation of the record with itself shifted by m steps. It equals 1 at m = 0 and lies between −1 and 1. A fast decay of ρ(m) to zero means a short memory, and the lag at which ρ(m) first drops below 1/e ≈ 0.37 is a common estimate of the correlation time. A record with a periodic component has an autocorrelation that oscillates with the same period. For N independent values, ρ(m) at m > 0 fluctuates around zero with a standard deviation of about $1/\sqrt N$, so values inside $\pm2/\sqrt N$ are compatible with no memory at all. When the lag equals a forecast horizon τ we write the same function as ρ(τ), the correlation between the value of today and the value τ steps later.

The power spectrum tells which periods are present in the record. The discrete Fourier transform writes the record as a sum of sinusoids with the frequencies $f_j=j/(N\Delta t)$, j = 0, 1, …, N/2, where Δt is the sampling interval,
$$X_j=\sum_{k=0}^{N-1}(s_k-\bar s)\,e^{-2\pi i jk/N} . \tag{3}$$
The magnitude $|X_j|$ says how strongly the frequency $f_j$ is present. The periodogram
$$P_j=\frac{\Delta t}{N}\,|X_j|^2 \tag{4}$$
is the simplest estimate of the power spectrum, the share of the variance of the record per unit of frequency. A peak at the frequency $f_j$ marks a periodic or nearly periodic component with the period $1/f_j$. The frequency resolution is $1/(N\Delta t)$, so a record of 100 years resolves frequencies to one cycle per 100 years. The periodogram fluctuates strongly from one frequency to the next, and the Welch method reduces this by cutting the record into overlapping segments, computing the periodogram of each and averaging, at the price of a coarser resolution. The power spectrum and the autocorrelation function carry the same information, since by the Wiener–Khinchin theorem one is the Fourier transform of the other.

The delay plot places every value $s_k$ against the value κ steps earlier, $s_{k-\kappa}$. This gives a two-dimensional delay embedding. A record from a low-dimensional deterministic law traces a thin curve or a structured band, a periodic record a closed loop, and a record without memory a round cloud.

We compute the autocorrelation of equation (2) and the periodogram of equation (4) with NumPy and check both against `statsmodels` and `scipy`.

In [ ]:
def autocorrelation(x, max_lag):
    # equation (2) for m = 0, 1, ..., max_lag
    xc = x - x.mean()
    den = np.sum(xc ** 2)
    return np.array([np.sum(xc[:len(x) - m] * xc[m:]) / den for m in range(max_lag + 1)])

def periodogram(x, dt):
    # equations (3) and (4): frequencies f_j = j / (N dt) and P_j = dt |X_j|^2 / N
    X = np.fft.rfft(x - x.mean())
    f = np.fft.rfftfreq(len(x), d=dt)
    return f, dt * np.abs(X) ** 2 / len(x)

def correlation_time(rho):
    # first lag at which the autocorrelation drops below 1/e
    below = np.flatnonzero(rho < np.exp(-1))
    return int(below[0]) if below.size else None

_x = np.cumsum(np.random.default_rng(0).standard_normal(500)) * 0.1 + np.random.default_rng(1).standard_normal(500)
assert np.allclose(autocorrelation(_x, 30), acf(_x, nlags=30, fft=False))
_f, _P = periodogram(_x, 1.0)
_fs, _Ps = scipy.signal.periodogram(_x - _x.mean(), fs=1.0, scaling="density", detrend=False)
assert np.allclose(_P[1:-1], _Ps[1:-1] / 2)       # scipy counts both signs of the frequency in the one-sided density
print("autocorrelation agrees with statsmodels, periodogram agrees with scipy")

<hr/>
## 3. Niño 3.4 variations

The plot of the whole record shows how often and how strongly El Niño and La Niña occur, and whether the record changes its character over the 150 years. The zoom on the years since 1980 shows the shape of single events. The dotted lines mark the thresholds of ±0.5 °C.

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(13, 7.5), layout="constrained")
axs[0].plot(t_enso, nino, color=UJBlue, lw=0.7)
for ax in axs:
    ax.axhline(0.5, color=DSOrange, lw=0.8, ls=":"); ax.axhline(-0.5, color=UJNavy, lw=0.8, ls=":")
    ax.set_ylabel("Niño 3.4 anomaly [°C]")
sel = t_enso >= 1980
axs[1].plot(t_enso[sel], nino[sel], color=UJBlue, lw=1.2)
axs[1].set_xlabel("year")
plt.show()
print(f"mean {nino.mean():.2f} °C, standard deviation {nino.std():.2f} °C, min {nino.min():.2f} in {t_enso[nino.argmin()]:.2f}, "
      f"max {nino.max():.2f} in {t_enso[nino.argmax()]:.2f}")
peaks = [(int(t_enso[i]), round(nino[i], 2)) for i in np.argsort(nino)[::-1] if True][:40]
big = {}
for y, v in peaks:
    if all(abs(y - z) > 1 for z in big) and len(big) < 5:
        big[y] = v
print("five warmest episodes (year of the monthly maximum, °C):", big)
print(f"fraction of months above +0.5 °C: {np.mean(nino > 0.5):.2f}, below -0.5 °C: {np.mean(nino < -0.5):.2f}")

The histogram shows how the values are distributed. Its shape decides whether a mean squared error, which weights large deviations strongly, is a sensible loss, and whether the record has hard bounds or rare extreme values that a forecast has to handle. The skewness measures the asymmetry of the distribution, zero for a symmetric one.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.3), layout="constrained")
ax.hist(nino, bins=50, color=UJBlue, alpha=0.8)
ax.set_xlabel("Niño 3.4 anomaly [°C]"); ax.set_ylabel("number of months")
plt.show()
print(f"skewness {scipy.stats.skew(nino):.2f}, excess kurtosis {scipy.stats.kurtosis(nino):.2f}")

The autocorrelation function of equation (2) measures how long correlations persist in the record. Their duration sets the scale of the forecasts. Persistence reaches about $\rho(\tau)^2$ at the horizon τ, so the decay of ρ shows how fast the simplest forecast loses its skill, and an oscillation of ρ reveals a cycle that a model can continue.

In [ ]:
MAX_LAG = 12 * 10
rho = autocorrelation(nino, MAX_LAG)
lags_years = np.arange(MAX_LAG + 1) * DT
fig, ax = plt.subplots(figsize=(11, 4.3), layout="constrained")
ax.plot(lags_years, rho, color=UJBlue)
ax.axhspan(-2 / np.sqrt(len(nino)), 2 / np.sqrt(len(nino)), color="0.85", zorder=0)
ax.axhline(np.exp(-1), color=DSOrange, ls=":", lw=1)
ax.set_xlabel("lag [years]"); ax.set_ylabel(r"$\rho(m)$")
plt.show()
first_min = np.argmin(rho[:12 * 4])
print(f"correlation time (rho < 1/e): {correlation_time(rho)} months")
print(f"rho at 1, 3, 6, 12, 20 months: {np.round(rho[[1, 3, 6, 12, 20]], 3)}")
print(f"first minimum at {first_min} months (rho = {rho[first_min]:.3f})")

The power spectrum of equation (4) decomposes the variance of the record into periods. A sharp peak marks a cycle with a fixed period, a broad hump an irregular oscillation, and a flat spectrum a record without memory. We plot the raw periodogram and its Welch average on a logarithmic power axis.

In [ ]:
f, P = periodogram(nino, DT)
f_w, P_w = scipy.signal.welch(nino - nino.mean(), fs=1 / DT, nperseg=12 * 32, scaling="density")
j_peak = 1 + np.argmax(P_w[1:])
fig, ax = plt.subplots(figsize=(11, 4.6), layout="constrained")
ax.semilogy(f[1:], P[1:], color="0.6", lw=0.8, label="periodogram")
ax.semilogy(f_w[1:], P_w[1:] / 2, color=UJBlue, lw=2, label="Welch, segments of 32 years")
for per in (2, 7):
    ax.axvline(1 / per, color=DSOrange, ls=":", lw=1.2)
ax.set_xlim(0, 2.0); ax.set_xlabel("frequency [cycles per year]"); ax.set_ylabel("power spectral density")
ax.legend()
plt.show()
print(f"record length {len(nino) * DT:.1f} years, frequency resolution {1 / (len(nino) * DT):.4f} cycles per year")
print(f"largest peak of the Welch spectrum at f = {f_w[j_peak]:.3f} per year, period {1 / f_w[j_peak]:.2f} years")
band = (f > 1 / 7) & (f < 1 / 2)
print(f"fraction of the variance at periods between 2 and 7 years: {P[band].sum() / P[1:].sum():.2f}")

The delay plot draws each value against the value κ samples earlier. It represents both linear and nonlinear dependence between past and future through a two-dimensional delay embedding. We draw it for a short, an intermediate and a long delay.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4.8), layout="constrained")
for ax, kappa in zip(axs, (1, 6, 18)):
    ax.plot(nino[kappa:], nino[:-kappa], ".", ms=2, alpha=0.5, color=UJBlue)
    ax.set_xlabel(r"$s_k$ [°C]"); ax.set_ylabel(rf"$s_{{k-{kappa}}}$ [°C]"); ax.set_title(rf"$\kappa = {kappa}$ months")
plt.show()

### Delay embedding

The hidden state of a deterministic system can be reconstructed from one measured coordinate using the delay vector
$$\mathbf u_k=\big(s_k,\;s_{k-\kappa},\;\dots,\;s_{k-(m-1)\kappa}\big) , \tag{5}$$
built from m values of the record spaced by the delay κ. By the theorem of Takens, for a large enough embedding dimension m the cloud of delay vectors is a faithful picture of the attractor. Mutual information determines κ from the dependence between delayed values, and false nearest neighbours determine m from the geometry of the reconstructed record.

The delay κ is taken at the first local minimum of the mutual information between $s_k$ and $s_{k-\kappa}$ [A. M. Fraser, H. L. Swinney, Phys. Rev. A **33**, 1134 (1986), [doi:10.1103/PhysRevA.33.1134](https://doi.org/10.1103/PhysRevA.33.1134)]. The range of the record is divided into 16 bins of equal width. For a delay κ we count, over all pairs $(s_{k-\kappa},s_k)$, the fraction $p_{ij}(\kappa)$ of pairs whose first value falls into bin i and second value into bin j, and the fractions $p_i$ and $p_j$ of single values in each bin. The mutual information is
$$I(\kappa)=\sum_{i,j}p_{ij}(\kappa)\,\ln\frac{p_{ij}(\kappa)}{p_i\,p_j} , \tag{6}$$
measured in nats. It is zero when the two values are independent, since then $p_{ij}=p_ip_j$ and every logarithm vanishes, and it grows with the dependence of the two values, linear or nonlinear. The autocorrelation of equation (2) measures only the linear part of this dependence. At the first minimum of I(κ) a new delay coordinate adds the most new information while it still belongs to the same stretch of motion.

The embedding dimension m is found with false nearest neighbours [M. B. Kennel, R. Brown, H. D. I. Abarbanel, Phys. Rev. A **45**, 3403 (1992), [doi:10.1103/PhysRevA.45.3403](https://doi.org/10.1103/PhysRevA.45.3403)]. If m is too small, the picture of the attractor is folded onto itself and points from distant parts of the orbit appear as neighbours. For every delay vector $\mathbf u_i^{(m)}$ of dimension m we find its nearest neighbour $\mathbf u_j^{(m)}$, excluding the points with $|i-j|\le w$, which are close only because they are close in time. This exclusion is called the Theiler window. The pair is a false neighbour if the next delay coordinate separates the two points strongly,
$$\frac{\big|s_{i-m\kappa}-s_{j-m\kappa}\big|}{\big\|\mathbf u_i^{(m)}-\mathbf u_j^{(m)}\big\|}>R_{\rm tol}=15 . \tag{7}$$
The fraction of false neighbours is computed for m = 1, 2, …, and the embedding dimension is the smallest m at which it falls below 5%. The ratio test alone is unreliable at large m. The distance between nearest neighbours grows with m, because every added coordinate adds to it, and the ratio of equation (7) becomes small for any record, also for noise. Kennel and co-workers therefore add an absolute test. The pair is also counted as false if its distance in m + 1 dimensions exceeds twice the standard deviation $\sigma_s$ of the record,
$$\frac{\Big(\big\|\mathbf u_i^{(m)}-\mathbf u_j^{(m)}\big\|^2+\big|s_{i-m\kappa}-s_{j-m\kappa}\big|^2\Big)^{1/2}}{\sigma_s}>2 , \tag{8}$$
which identifies neighbours separated by a large distance relative to the variation in the record. We compute both fractions and choose the embedding dimension with the ratio test. The combined fraction, which counts a pair as false under the ratio test or under the absolute test, distinguishes deterministic from stochastic records. For a low-dimensional deterministic law it stays near zero once the attractor is unfolded. For a stochastic record new random influences enter every coordinate, the nearest neighbours stay far apart in the sense of equation (8), and the combined fraction stays high and grows with m.

The embedding window has the length $(m-1)\kappa+1$ samples and can serve directly as the input of the AR model and of NVAR, with L = m values taken with the stride κ.

### Exercise 1
1. Run the exploration cells and describe the record, its range, the frequency of El Niño and La Niña conditions, the shape of the distribution and the strongest events.
2. Read off the correlation time from the autocorrelation function, and the range of periods from the power spectrum. Compare the regularity of ENSO with the regularity of the sunspot cycle.
3. Explain the shapes of the delay plots, and state up to which horizon you expect a forecast from the last 20 months to work.
4. Compute the mutual information of equation (6) up to a delay of 48 months and find the delay κ at its first minimum. Compute the fraction of false neighbours of equation (7) for m = 1, …, 8 with a Theiler window of 7 months and find the embedding dimension m, following the notebook on the weather in central England.

In [ ]:
SERIES = nino          # the record that the forecasting protocol below splits into blocks

<hr/>
## 4. Forecasting protocol and scores

The record of length N is split in time order into a fitting block, the first 60% of the samples, a validation block, the next 20%, and a test block, the last 20%. A model is fitted on the fitting block, its settings are chosen by the score on the validation block, the chosen model is fitted again on the fitting and validation blocks together, and it is scored once on the test block. The record is standardised with the mean and the standard deviation of its fitting block. The split follows the time order because a forecast is always made from the past into the future. A random split would put neighbouring, strongly correlated values into the fitting and the test blocks, and the test score would measure how well a model interpolates between known values. Settings chosen on the test block would make its score optimistic, and a separate validation block is therefore used to choose them.

From every origin k the models receive the values up to $s_k$, at most the last L = 20 of them, and forecast $\hat s_{k+\tau}$ for the horizons τ = 1, …, 20, in months. The score at horizon τ is the forecast capacity, the squared Pearson correlation between forecasts and targets over all test origins,
$$C(\tau)=r(\tau)^2,\qquad r(\tau)=\frac{\sum_k\big(\hat s_{k+\tau}-\overline{\hat s}_\tau\big)\big(s_{k+\tau}-\overline s_\tau\big)}{\sqrt{\sum_k\big(\hat s_{k+\tau}-\overline{\hat s}_\tau\big)^2\,\sum_k\big(s_{k+\tau}-\overline s_\tau\big)^2}} , \tag{9}$$
where the bars are averages over the origins at fixed τ. C(τ) is the fraction of the variance of the targets explained by a straight-line fit on the forecasts. It is 1 when the forecasts lie on any straight line against the targets and 0 when they are uncorrelated. The total capacity
$$C_{\rm tot}=\sum_{\tau=1}^{20}C(\tau) \tag{10}$$
has the maximum 20. A forecast without skill still gives C(τ) of about 1/n for n test origins.

Every model is compared with persistence, the forecast that the record keeps its last value,
$$\hat s_{k+\tau}=s_k\quad\text{for every }\tau . \tag{11}$$
Persistence needs no fitting and uses only the newest value. Its capacity follows from the autocorrelation function of equation (2). The correlation between $s_k$ and $s_{k+\tau}$ over the test origins is close to the autocorrelation ρ(τ) of the record at the lag τ, so persistence reaches $C(\tau)\approx\rho(\tau)^2$. A model is useful only where it beats persistence, and the gain over persistence measures what the model has learned beyond the memory of the last value.

In [ ]:
TAU_MAX, L_MAX = 20, 20
TAUS = np.arange(1, TAU_MAX + 1)
N = len(SERIES)                                    # length of the record used for forecasting
N_FIT, N_VAL = int(0.6 * N), int(0.2 * N)
K0 = L_MAX - 1                                     # first origin with a full window of L_MAX values
O_FIT = np.arange(K0, N_FIT - TAU_MAX)             # origins whose targets lie in the fitting block
O_VAL = np.arange(N_FIT, N_FIT + N_VAL - TAU_MAX)  # ... in the validation block
O_TRAIN = np.arange(K0, N_FIT + N_VAL - TAU_MAX)   # ... in fitting and validation blocks (final fit)
O_TEST = np.arange(N_FIT + N_VAL, N - TAU_MAX)     # ... in the test block
print(f"N = {N}: fitting block 0..{N_FIT - 1}, validation {N_FIT}..{N_FIT + N_VAL - 1}, test {N_FIT + N_VAL}..{N - 1}")
print(f"origins: fit {len(O_FIT)}, validation {len(O_VAL)}, train {len(O_TRAIN)}, test {len(O_TEST)}")
print(f"no-skill floor: 1/n = {1 / len(O_TEST):.4f} per horizon, {TAU_MAX / len(O_TEST):.3f} for C_tot")

def standardise(x):
    # mean and standard deviation of the fitting block only
    return (x - x[:N_FIT].mean()) / x[:N_FIT].std()

def windows(s, origins, L):
    # rows (s_k, s_(k-1), ..., s_(k-L+1)), newest value first; shape (len(origins), L)
    return s[np.asarray(origins)[:, None] - np.arange(L)[None, :]]

def targets(s, origins):
    # Y[i, tau-1] = s_(k_i + tau); shape (len(origins), TAU_MAX)
    return s[np.asarray(origins)[:, None] + TAUS[None, :]]

def capacity(P, Y):
    # C(tau) of equation (15) for every column; a constant forecast has r = 0
    Pc, Yc = P - P.mean(0), Y - Y.mean(0)
    den = np.sqrt((Pc ** 2).sum(0) * (Yc ** 2).sum(0))
    r = np.where(den > 0, (Pc * Yc).sum(0) / np.where(den > 0, den, 1.0), 0.0)
    return r ** 2

# check of capacity against scipy on random numbers
_P, _Y = np.random.default_rng(0).standard_normal((2, 300, 3))
assert np.allclose(capacity(_P, _Y), [scipy.stats.pearsonr(_P[:, j], _Y[:, j]).statistic ** 2 for j in range(3)])

We divide the record in time order into fitting, validation and test blocks. At an origin k, the model receives a window of the last L values and forecasts the value τ steps later. The input window is marked in blue and the target in orange. We repeat the forecast at every origin of a block and compare the predictions with the measured values.

In [ ]:
# sketch of the forecasting task: blocks, window of past values, forecast at the horizon tau
rng = np.random.default_rng(1)
t = np.arange(120)
x = np.sin(2 * np.pi * t / 37) + 0.3 * rng.standard_normal(120)
k, L_demo, tau_demo = 108, 12, 6
fig, ax = plt.subplots(figsize=(13, 4.2), layout="constrained")
for (a, b), name, col in (((0, 72), "fitting block", "#DCE6F2"), ((72, 96), "validation block", "#E8F1E4"),
                          ((96, 120), "test block", "#FBE7DA")):
    ax.axvspan(a - 0.5, b - 0.5, color=col, zorder=0)
    ax.text((a + b) / 2, 2.25, name, ha="center", fontsize=13)
ax.plot(t, x, "o-", color="0.6", ms=3, lw=1)
win = np.arange(k - L_demo + 1, k + 1)
ax.plot(win, x[win], "o", color=UJBlue, ms=7, label=f"window of the last L = {L_demo} values")
ax.plot(k + tau_demo, x[k + tau_demo], "*", color=DSOrange, ms=18, label=rf"target $s_{{k+\tau}}$, $\tau$ = {tau_demo}")
ax.annotate("", xy=(k + tau_demo, 1.6), xytext=(k, 1.6), arrowprops=dict(arrowstyle="->", lw=2, color=DSOrange))
ax.text(k + tau_demo / 2, 1.72, r"horizon $\tau$", ha="center", color=DSOrange, fontsize=13)
ax.axvline(k, color=UJNavy, ls=":", lw=1.5); ax.text(k, -2.3, "origin $k$", ha="center", color=UJNavy, fontsize=13)
ax.set_ylim(-3.2, 2.6); ax.set_xlabel("time step"); ax.set_ylabel("$s_k$"); ax.legend(loc="lower left", ncol=2)
plt.show()

For each signal, `plot_scatter` draws the forecasts of every model against the targets at the horizons τ = 1, 5 and 20, one row per model, with the diagonal $\hat s=s$ and the value of C(τ) in each panel. `plot_capacity` draws C(τ) of all models against τ, and `ctot_table` prints the total capacities.

In [ ]:
MODEL_COLORS = {"persistence": UJGray, "AR": UJBlue, "MA": "#7FB3E0", "ARMA": UJNavy, "NVAR": "#2A9D8F",
                "MLP": "#E9C46A", "RNN": "#B5651D", "LSTM": DSOrange}

def plot_scatter(name, preds, s, taus=(1, 5, 20)):
    Y = targets(s, O_TEST)
    models = list(preds)
    fig, axs = plt.subplots(len(models), len(taus), figsize=(3.4 * len(taus), 3.0 * len(models)),
                            layout="constrained", squeeze=False)
    lim = np.quantile(np.abs(Y), 0.995) * 1.1
    for i, m in enumerate(models):
        C = capacity(preds[m], Y)
        for j, t in enumerate(taus):
            ax = axs[i, j]
            ax.plot(Y[:, t - 1], preds[m][:, t - 1], ".", ms=3, alpha=0.5, color=MODEL_COLORS.get(m, UJBlue))
            ax.plot([-lim, lim], [-lim, lim], color="0.5", lw=0.8, ls="--")
            ax.set_xlim(-lim, lim)
            ax.text(0.04, 0.96, f"$C={C[t - 1]:.3f}$", transform=ax.transAxes, va="top", fontsize=12)
            if i == 0: ax.set_title(rf"$\tau={t}$")
            if i == len(models) - 1: ax.set_xlabel(r"true $s_{k+\tau}$")
            if j == 0: ax.set_ylabel(f"{m}\n" + r"forecast $\hat s_{k+\tau}$")
    fig.suptitle(name)
    plt.show()

def plot_capacity(results, names, theory=None, ylim=None):
    fig, axs = plt.subplots(1, len(names), figsize=(6.2 * len(names), 4.6), layout="constrained", squeeze=False)
    for ax, name in zip(axs[0], names):
        for m, C in results[name].items():
            ax.plot(TAUS, C, "o-", ms=4, color=MODEL_COLORS.get(m, UJBlue), label=m)
        if theory is not None and name in theory:
            ax.plot(TAUS, theory[name], "k--", lw=1.5, label="theory")
        ax.axhline(1 / len(O_TEST), color="0.6", lw=1, ls=":")
        ax.set_title(name); ax.set_xlabel(r"horizon $\tau$")
        if ylim: ax.set_ylim(*ylim)
    axs[0, 0].set_ylabel(r"$C(\tau)$"); axs[0, -1].legend(fontsize=11, loc="upper right")
    plt.show()

def ctot_table(results):
    tab = pd.DataFrame({name: {m: C.sum() for m, C in res.items()} for name, res in results.items()})
    print("total capacity C_tot = sum of C(tau) over tau = 1..20 (maximum 20)")
    print(tab.round(3).to_string(na_rep="-"))
    return tab

<hr/>
## 5. Models

The models form a sequence of increasing flexibility. Persistence uses only the last value. AR, MA and ARMA are linear in the past values or past errors. NVAR adds products of past values and is still fitted by linear least squares. The MLP learns its own nonlinear features of the window, and the RNN and the LSTM carry a hidden state through the window. Comparing them on the same record shows how much of the future is fixed by linear memory alone and how much more a nonlinear model can extract.

Every model is a function that receives a standardised signal s and returns the forecasts for all test origins and all horizons, an array of shape (number of test origins, 20). Persistence, which forecasts every horizon by the last value $s_k$, is given as an example. The other seven functions are yours to write. The notebooks on the weather in central England and on the sunspot number apply the same seven models with the same settings to their records and contain complete implementations to start from, and the lecture notebooks explain every model in detail.

1. **AR**, the direct autoregressive model. For every horizon τ a linear readout of the window $(s_k,\dots,s_{k-L+1})$ plus an offset, fitted by ridge regression, all horizons at once as the columns of one target matrix. Choose L from {1, 2, 3, 5, 10, 20} and the ridge strength λ from {10⁻⁶, 10⁻², 1, 100} by $C_{\rm tot}$ on the validation block.
2. **MA**, the moving-average model MA(q), fitted by maximum likelihood with `statsmodels` (`ARIMA` with order (0, 0, q) and a constant). The τ-step forecast iterates the one-step model with the future errors set to zero. The past errors $e_k$ are the residuals of the fitted model on the whole record (`res.apply(s).resid`). Choose q from {1, 2, 3, 5} on the validation block.
3. **ARMA**, the model ARMA(p, q), fitted and iterated in the same way, with (p, q) from {(1, 1), (2, 1), (1, 2), (2, 2)}.
4. **NVAR**, the window of L values with stride κ, every κ-th value, together with all products of two of its entries, and a ridge readout per horizon. Choose L from {2, 3, 5, 8}, κ from {1, 2, 4} and λ as for AR.
5. **MLP**, `sklearn.neural_network.MLPRegressor` with tanh units on the window of the last 20 values, with 20 outputs, one per horizon. Choose one hidden layer of 32 or 64 units and the weight penalty `alpha` from {10⁻³, 10⁻¹, 1} on the validation block, with `early_stopping=True` and `random_state=0`.
6. **RNN**, an Elman recurrent network with 32 hidden units, run over the window of the last 20 values from the oldest to the newest, with a linear readout of the last hidden state to the 20 horizons. Train it with Adam on the mean squared error of the fitting block, with gradient clipping and early stopping on the validation loss.
7. **LSTM**, the same with an LSTM cell.

The direct AR model and NVAR both fit their readouts by ridge regression. For a feature matrix Φ with one row per origin and a target matrix Y with one column per horizon, the ridge loss is
$$\mathcal L(W,\mathbf b)=\big\|Y-\Phi W-\mathbf 1\mathbf b^\top\big\|^2+\lambda\|W\|^2 , \tag{12}$$
where $\mathbf 1$ is a column of ones and $\mathbf b$ the offsets of the horizons. Its minimum follows from the centred matrices $\Phi_c$ and $Y_c$, from which the column means are subtracted,
$$W=(\Phi_c^\top\Phi_c+\lambda I)^{-1}\Phi_c^\top Y_c,\qquad \mathbf b^\top=\overline Y-\overline\Phi\,W , \tag{13}$$
where the bars are the column means, so the offsets are not penalised.

The MA and ARMA models are fitted by maximum likelihood, since their past errors depend on the parameters themselves. With the statsmodels convention, the fitted ARMA(p, q) model reads
$$s_k-\mu=\sum_{i=1}^p\phi_i\,(s_{k-i}-\mu)+e_k+\sum_{j=1}^q\theta_j\,e_{k-j} , \tag{14}$$
where μ is the mean of the process and $e_k$ the one-step error. The τ-step forecast applies equation (14) repeatedly from the origin k, with the forecasts in place of the unknown future values and zero in place of the unknown future errors. An MA(q) forecast therefore returns to μ after q steps.

The functions receive only the signal. Use the origins `O_FIT`, `O_VAL`, `O_TRAIN` and `O_TEST` and the functions `windows`, `targets` and `capacity` of Section 4.

In [ ]:
CHOSEN = {}          # settings chosen on the validation block, filled in by the forecasters

def forecast_persistence(s):
    # s_(k+tau) is forecast by s_k for every horizon
    return np.repeat(s[O_TEST][:, None], TAU_MAX, axis=1)

In [ ]:
def forecast_ar(s):
    """Direct AR: ridge readout of the window, one column per horizon; L and lambda on the validation block.
    Returns an array of shape (len(O_TEST), TAU_MAX)."""
    raise NotImplementedError

def forecast_ma(s):
    """MA(q) fitted with statsmodels, iterated with future errors set to zero; q on the validation block.
    Returns an array of shape (len(O_TEST), TAU_MAX)."""
    raise NotImplementedError

def forecast_arma(s):
    """ARMA(p, q) fitted with statsmodels, iterated with future errors set to zero; (p, q) on the validation block.
    Returns an array of shape (len(O_TEST), TAU_MAX)."""
    raise NotImplementedError

def forecast_nvar(s):
    """NVAR: window with stride, all quadratic products, ridge readout; L, stride and lambda on the validation block.
    Returns an array of shape (len(O_TEST), TAU_MAX)."""
    raise NotImplementedError

def forecast_mlp(s):
    """MLPRegressor on the window of L_MAX values, 20 outputs; width and alpha on the validation block.
    Returns an array of shape (len(O_TEST), TAU_MAX)."""
    raise NotImplementedError

def forecast_rnn(s):
    """Elman RNN with 32 units over the window, readout of the last hidden state; early stopping on the validation loss.
    Returns an array of shape (len(O_TEST), TAU_MAX)."""
    raise NotImplementedError

def forecast_lstm(s):
    """LSTM with 32 units over the window, readout of the last hidden state; early stopping on the validation loss.
    Returns an array of shape (len(O_TEST), TAU_MAX)."""
    raise NotImplementedError

The function `run_models` applies every implemented model to a standardised record, scores the forecasts with equation (9), stores the capacities and the forecasts, and prints the settings chosen on the validation block.

In [ ]:
FORECASTERS = {"persistence": forecast_persistence, "AR": forecast_ar, "MA": forecast_ma, "ARMA": forecast_arma,
               "NVAR": forecast_nvar, "MLP": forecast_mlp, "RNN": forecast_rnn, "LSTM": forecast_lstm}
RESULTS, PREDS = {}, {}

def run_models(name, s, models=tuple(FORECASTERS)):
    # runs every implemented model on the standardised signal s; unimplemented ones are skipped
    RESULTS[name], PREDS[name] = {}, {}
    Y = targets(s, O_TEST)
    for m in models:
        t0 = time.time()
        try:
            P = FORECASTERS[m](s)
        except NotImplementedError:
            continue
        PREDS[name][m], RESULTS[name][m] = P, capacity(P, Y)
        info = CHOSEN.get(m, "")
        print(f"{name:14s} {m:12s} C_tot {RESULTS[name][m].sum():7.3f}   C(1) {RESULTS[name][m][0]:.3f}   "
              f"{info}  ({time.time() - t0:.1f} s)")

<hr/>
## 6. Forecasts of the Niño 3.4 index

All eight models are fitted to the standardised record and scored on the test block. The scatter plots at 1, 5 and 20 months show systematic errors, the curves C(τ) show how fast each model loses its skill with the horizon, and the table of total capacities ranks the models. Persistence is the reference that every model has to beat.

### Exercise 2
1. Run all models on the standardised Niño 3.4 record. Plot the scatter of forecasts against targets at τ = 1, 5, 20 months, plot C(τ) for all models and print the table of total capacities.
2. Compare every model with persistence, and relate C(τ) to the autocorrelation function of Section 3. Find the horizon at which C(τ) of the best model drops below 0.5.
3. Look at the forecasts of the strongest El Niño in the test block and describe how early the models see it coming.
4. Fit AR and NVAR on the embedding window of Exercise 1, with L = m values taken with the stride κ, and compare their total capacities with the windows chosen on the validation block.

In [ ]:
s_enso = standardise(nino)
print(f"test block: {t_enso[N_FIT + N_VAL]:.2f} to {t_enso[-1]:.2f}")
run_models("Niño 3.4", s_enso)

In [ ]:
plot_scatter("Niño 3.4 anomaly", PREDS["Niño 3.4"], s_enso)
plot_capacity(RESULTS, ["Niño 3.4"], ylim=(0, 1.02))
ctot_table(RESULTS);

### Exercise 3
1. Write a short report of one page. Describe the Niño 3.4 record with the tools of Section 2, state which models forecast it best and up to which horizon, and explain the result with the properties found in the exploration.
2. Compare the results with the notebook on the sunspot number, and explain with the autocorrelation functions and the spectra of the two records why one of them is easier to forecast.

## References

- NOAA Physical Sciences Laboratory, *Niño 3.4 SST anomaly from HadISST*, <https://psl.noaa.gov/data/timeseries/month/>
- N. A. Rayner, D. E. Parker, E. B. Horton, C. K. Folland, L. V. Alexander, D. P. Rowell, E. C. Kent, A. Kaplan, *Global analyses of sea surface temperature, sea ice, and night marine air temperature since the late nineteenth century*, J. Geophys. Res. **108**, 4407 (2003), [doi:10.1029/2002JD002670](https://doi.org/10.1029/2002JD002670)
- D. J. Gauthier, E. Bollt, A. Griffith, W. A. S. Barbosa, *Next generation reservoir computing*, Nat. Commun. **12**, 5564 (2021), [doi:10.1038/s41467-021-25801-2](https://doi.org/10.1038/s41467-021-25801-2)